### **TALLEER 10 - Proyecto #1 - Análisis y Visualización de Datos**

### Explicación del Notebook
**Problema:** Los valores atípicos y las diferencias de escala distorsionan el análisis y degradan el rendimiento de modelos supervisados.

**Objetivo general:** Analizar el dataset y aplicar una estrategia sistemática de detección y corrección de outliers para obtener una vista minable de mayor calidad.

**Objetivos específicos:**
1. Explorar variables y diagnosticar distribución, correlaciones y presencia de valores atípicos.
2. Corregir outliers superiores e inferiores usando criterios cuantílicos/IQR de forma trazable.
3. Escalar variables y dejar el conjunto de datos listo para entrenamiento y evaluación de modelos.

##**1.1 - Cargar Librerias de Preprocesamiento de Datos**

In [ ]:
#========================================
#  Libreria para tratamiento de datos
#========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
#========================================
#  Libreria para escalamiento o normalizacion de datos
#========================================
from sklearn.preprocessing import StandardScaler

In [ ]:
#========================================
#  Libreria para division de la dataset en datos de entrenamiento y datos de pruebas
#========================================
from sklearn.model_selection import train_test_split


In [ ]:
#========================================
#  Libreria para graficos profesionales
#========================================
import matplotlib.pyplot as plt
import seaborn as sns

##**1.2. Cargamos Librerías para problemas de Regresión (variable Y continua)**

In [ ]:
#========================================
#  Librerias para Algoritmo de Regresión Lineal
#  Sirve para realizar modelos que van a predecir el valor
#  de una variable Y que es de naturaleza numerica continua
#========================================
from sklearn.linear_model import LinearRegression

In [ ]:
#========================================
#  Libreria de Maquina de Soporte Vectorial SVM-R para problemas de Regresión
#========================================
# Entrenamiento Soporte Vectorial para Regresión
from sklearn.svm import SVR

##**1.3. Cargamos Librerías para problemas de Clasificación (variable Y discreta)**

In [ ]:
#========================================
# Libreria de Algoritmo de Regresión Logistica
#========================================
#  Sirve para crear modelos que van a predecir el valor
#  de una variable Y que es de naturaleza numerica discreta
# Entrenamiento Logistico
from sklearn.linear_model import LogisticRegression

In [ ]:
#========================================
#  Libreria de Maquina de Soporte Vectorial SVM-C para problemas de clasificación
#========================================
# Entrenamiento Soporte Vectorial para Clasificación
from sklearn.svm import SVC

##**1.4 - Cargamos librerías para Análisis de correlación entre variables Xs, y variables Xs con la Y**

In [ ]:
#========================================
#  Libreria para Analisis de correlacion entre Xs y Y
#========================================
import statsmodels.api as sm

##**1.5 - Cargamos librerías para medir la calidad del entrenamiento de los modelos de Aprendizaje Supervisado para clasificación**

In [ ]:
#========================================
# Cargamos librerias para medir la calidad del entrenamiento de los modelos de clasificación
#========================================
from sklearn.metrics import confusion_matrix, accuracy_score

#**2 - Cargar los Datos**

In [ ]:
# Obtenemos los datos y los cargamos en la carpeta Dataset1 de Google Drive
# Dataset: DatosEncuestasDietaFinal_v2.xlsx

##**2.1 - Cargamos la librería para conectarnos al goggle drive**

In [ ]:
# Cargamos la libreria para conecatrnos al goggle drive
from google.colab import drive
drive.mount('/content/drive')

##**2.2 - Recuperamos la ruta donde esta cargado el dataset en Google Drive**

In [ ]:
# import pandas as pd
# /content/drive/MyDrive/Datasets1/TALLER_10_Dataset_EncuestasDieta_v3.xlsx

ruta_dataset = "/content/drive/MyDrive/Datasets1/TALLER_10_Dataset_EncuestasDieta_v3.xlsx"

df1_pacientes = pd.read_excel(ruta_dataset)

#**3 - Explorar los datos**

##**3.1 - Identificamos las variables disponibles**

In [ ]:
print('Variables de estudio disponibles en el dataframe:')
print(df1_pacientes.keys())

In [ ]:
# Variable dummy o innecesaria que se puede retirar
# id' - Secuencial numerico

#----------------------------------------
# Posibles Variables Xs
# 'sex' es el sexo del paciente
# 'age' Edad del paciente
# 'edc' Nivel de educacion
# 'wgt' Peso
# 'hgt' Estatura
# 'eat_seq' Numero de comidas que se sirve 1 a 7
# 'meal_type'  Tipo de alimento que se sirvio

#---------------------------------------------
# Posibles variavbles Ys
# 'bmi_cat'          Indice de Masa corporal  Y1
# 'energy_100_truc'  Cantida de Calorias que ha ingerido Y2


In [ ]:
# Visualizamos la estructura interna que uso python para cargar los datos
df1_pacientes.info()

In [ ]:
# Visualizamos una muestra de los datos al inicio
df1_pacientes.head(10)

In [ ]:
# Visualizamos una muestra de los datos al final del dataframe
df1_pacientes.tail(10)

In [ ]:
# Generamos una copia del dataframe para no alterar el original
df1_pacientes_cp = df1_pacientes.copy()

In [ ]:
# Retiramos variables innecesarias para este proyecto
# df1_pacientes_cp.drop(   'id'  , axis=1, inplace=True)

df1_pacientes_cp = df1_pacientes_cp.drop( ['id', 'energy_100_truc' ], axis=1)

In [ ]:
# Visualizamos el datafreame con las columnas a usar en el resto del análisis
df1_pacientes_cp.head(10)

In [ ]:
df1_pacientes_cp.describe()

In [ ]:
# Identificacion de datos vacios variables edc, bmi_cat
df1_pacientes_cp.info()

In [ ]:
df1_pacientes.shape

##**3.2 - Retiramos los registros cuya variable y esta vacia porque no sirven para entrenar el modelo**

In [ ]:
df1_pacientes_cp = df1_pacientes_cp[df1_pacientes['bmi_cat'].notna()]

In [ ]:
df1_pacientes_cp = df1_pacientes_cp[df1_pacientes['edc'].notna()]

In [ ]:
# Identificacion de datos vacios variables edc, bmi_cat
df1_pacientes_cp.info()

### Tabla de Clasificación de Status de Peso por Indice de Masa Corporal (BMI)

**bmi_cat** |**Weight Status**|  | **Adult BMI**|  
------------|----------------|--|---------------
1           |Underweight     |  | < 18.5       
2           |Normal weight   |  | 18.5 a 24.9    
3           |Overweight      |  | 25 a 29.9     
4           |Obesity         |  |  >= 30       

In [ ]:
# Inspeccion de tipos de Categorias BMI empleadas por encuestadores
df1_pacientes_cp = df1_pacientes_cp.sort_values('bmi_cat')
df1_pacientes_cp['bmi_cat'].unique()

In [ ]:
# Calculo de Matriz de Correlacion
df1_pacientes_cp.corr()

In [ ]:
# Ilustracion Grafica de Matriz de Correlacion
import seaborn as sns
sns.heatmap(df1_pacientes_cp.corr( ), vmin = -1, vmax = +1, annot = True, cmap = 'coolwarm')

In [ ]:
# Ilustracion Grafica de Matriz de Correlacion
import seaborn as sns
sns.heatmap(df1_pacientes_cp.corr( method='pearson', min_periods=1, numeric_only=False ), vmin = -1, vmax = +1, annot = True, cmap = 'coolwarm')

In [ ]:
# Ilustracion Grafica de Matriz de Correlacion
import seaborn as sns
sns.heatmap(df1_pacientes_cp.corr( method='kendall', min_periods=1, numeric_only=False ), vmin = -1, vmax = +1, annot = True, cmap = 'coolwarm')

In [ ]:
# Ilustracion Grafica de Matriz de Correlacion
import seaborn as sns
sns.heatmap(df1_pacientes_cp.corr( method='spearman', min_periods=1, numeric_only=False ), vmin = -1, vmax = +1, annot = True, cmap = 'coolwarm')

In [ ]:
# Visuaizacion Grafica de Histograma de Edades para identificar si la muestra
# muesta distribucion normal en los datos con las diveras edades esperadas.

count_ages = df1_pacientes_cp['age'].groupby(df1_pacientes_cp.age).agg('count')
namesbars = df1_pacientes_cp['age'].unique()
x_pos = np.arange(len(namesbars))
# Create bars and choose color
plt.bar(x_pos, count_ages)
# Add title and axis names
plt.title('Distribucion de Edades de los sujetos de estudio')
plt.xlabel('Edad')
plt.ylabel('Cantidad')
# Create names on the x axis
plt.xticks(x_pos, namesbars)
# Show graph
#plt.show()
plt.savefig('Grafico de Distribucion de Edades de la Muestra.png')

# 4 - Dar tramiento a los datos crudos para obtener una Vista Minable

##**4.1 Dividir los datos en entrada (Xs) y salida (Y)**

In [ ]:
# Obtencion de Variables Xs en un objeto numpy.ndarray
df_Xs = df1_pacientes_cp.iloc[  :  ,  0 : 7  ]

In [ ]:
type(df_Xs )

In [ ]:
df_Xs.head(5)

In [ ]:
# Obtencion de Variables Xs en un objeto numpy.ndarray
X = df1_pacientes_cp.iloc[  :  ,  0 : 7  ].values

In [ ]:
type(X)

In [ ]:
X

In [ ]:
# Opcion 1 usando una de las columnas del arreglo numpy
# le entregamos la columna de los datos sex en forma de lista extraida del arreglo numpy
plt.boxplot(X[:, 0])  # variable sex


In [ ]:
# Opcion 2 usando una de las columnas del dataframe
# le entregamos la columna de los datos sex usando el nombre de la variable
# eso logra que se extraiga la data en forma de lista extraida del dataframe

plt.boxplot(df_Xs["sex"])

In [ ]:
plt.boxplot(X[:, 1])  # variable age #SUPERIOR

In [ ]:
plt.boxplot(X[:, 2])  # variable edc, INFERIROR posee outlayer

In [ ]:
plt.boxplot(X[:, 3])  # variable wgt, posee outlayer SUPEIROR

In [ ]:
plt.boxplot(X[:, 4])  # variable hgt, posee outlayer superiores E INFERIOR

In [ ]:
plt.boxplot(X[:, 5])  # variable 'eat_seq', posee outlayer superiores e Inferiores

In [ ]:
plt.boxplot(X[:, 6])  # variable

##**4.2. Proceso de corrección de los Ouliers**

###**4.2.1 - Corrigiendo Outliers superiores**

In [ ]:
# ==========================================================================
# Tratamiento de Outliers Superiores (Valores atipicos, extremos superiores)
# ------ Se setea el indice de la columna con outlayer inferior
# ----- este bloque se ejeucta para cada variable con Outlayer inferior
# ----- col = 2da, 4ta, 5ta, 6ta
#      indice 1 ,  3 ,  4 ,  5
# ==========================================================================

In [ ]:
import numpy as np

In [ ]:
# ----- Funcion para rastreo del mejor percentil superior para ajuste de outlayers superiores
def HallaMejorPercentilSup(micol,  maximo, minimo, miCotaSup):
	for x in range(maximo, minimo, -1):
		valor_tope_actual = np.quantile(X[:,micol] , x/100)
		print(f"Percentil {x} = {np.round(valor_tope_actual,micol)},col = {micol}, CotaSup = {miCotaSup}")
		if(valor_tope_actual <= miCotaSup):
			return x , valor_tope_actual

In [ ]:
# ----- se deber ejecutar el bloque, ajustar el valor  de col y
# ------ ejecutarlo para cada variable con outlayer superior
# indice 1 ,  3 ,  4 ,  5

In [ ]:
col = 1
q3=np.quantile(X[:,col] , 0.75)
q1=np.quantile(X[:,col] , 0.25)

IQR=q3-q1

CotaSup= q3 + 1.5 * IQR

PercentilSupSugerido , ValorTecho = HallaMejorPercentilSup(col, 99, 80, CotaSup)

X[:,col] = np.where(X[:,col] > ValorTecho, ValorTecho, X[:,col] )

In [ ]:
plt.boxplot(X[:, 1])  # variable age #SUPERIOR

In [ ]:
col = 3
q3=np.quantile(X[:,col] , 0.75)
q1=np.quantile(X[:,col] , 0.25)
IQR=q3-q1
CotaSup=q3+1.5*IQR
PercentilSupSugerido , ValorTecho = HallaMejorPercentilSup(col, 99, 80, CotaSup)
X[:,col] = np.where(X[:,col] > ValorTecho, ValorTecho, X[:,col] )

In [ ]:
plt.boxplot(X[:, 3])

In [ ]:
col = 4
q3=np.quantile(X[:,col] , 0.75)
q1=np.quantile(X[:,col] , 0.25)
IQR=q3-q1
CotaSup=q3+1.5*IQR
PercentilSupSugerido , ValorTecho = HallaMejorPercentilSup(col, 99, 80, CotaSup)
X[:,col] = np.where(X[:,col] > ValorTecho, ValorTecho, X[:,col] )

In [ ]:
plt.boxplot(X[:, 4])

In [ ]:
col = 5
q3=np.quantile(X[:,col] , 0.75)
q1=np.quantile(X[:,col] , 0.25)
IQR=q3-q1
CotaSup=q3+1.5*IQR
PercentilSupSugerido , ValorTecho = HallaMejorPercentilSup(col, 99, 80, CotaSup)
X[:,col] = np.where(X[:,col] > ValorTecho, ValorTecho, X[:,col] )

In [ ]:
plt.boxplot(X[:, 5])

###**4.2.2 - Correccion de outlier inferiores**

In [ ]:
# =========================================================================
# Tratamiento de Outliers Inferiores (Valores atipicos, extremos Inferiores)
# ------ Se setea el indice de la columna con outlayer inferior
# ----- este bloque se ejeucta para cada variable con Outlayer inferior
#       columna 3ra 5ta  6ta
# ----- incide   2,  4,  5
# ==========================================================================

In [ ]:
# ----- Funcion creada para rastreo del mejor percentil inferior para ajuste de outlayers inferiores
def HallaMejorPercentilInf(micol,  minimo, maximo, miCotaInf):
  valor_tope_previo = 0
  valor_x_previo = 0
  for x in range(minimo, maximo, +1):
    valor_tope_actual = np.quantile(X[:,micol] , x/100)
    print(f"Percentil {x} = {np.round(valor_tope_actual,micol)},col = {micol}, CotaInf = {miCotaInf}")
    if(valor_tope_actual >= miCotaInf):
      return x, valor_tope_actual

In [ ]:
col = 2
q3=np.quantile(X[:,col] , 0.75)
q1=np.quantile(X[:,col] , 0.25)
IQR=q3-q1
CotaInf=q1-1.5*IQR
PercentilInfSugerido , ValorPiso = HallaMejorPercentilInf(col, 1, 20, CotaInf)
X[:,col] = np.where(X[:,col] < ValorPiso, ValorPiso, X[:,col] )

In [ ]:
plt.boxplot(X[:, 2])

In [ ]:
col = 4
q3=np.quantile(X[:,col] , 0.75)
q1=np.quantile(X[:,col] , 0.25)
IQR=q3-q1
CotaInf=q1-1.5*IQR
PercentilInfSugerido , ValorPiso = HallaMejorPercentilInf(col, 1, 20, CotaInf)
X[:,col] = np.where(X[:,col] < ValorPiso, ValorPiso, X[:,col] )

In [ ]:
plt.boxplot(X[:, 4])

In [ ]:
col = 5
q3=np.quantile(X[:,col] , 0.75)
q1=np.quantile(X[:,col] , 0.25)
IQR=q3-q1
CotaInf=q1-1.5*IQR
PercentilInfSugerido , ValorPiso = HallaMejorPercentilInf(col, 1, 20, CotaInf)
X[:,col] = np.where(X[:,col] < ValorPiso, ValorPiso, X[:,col] )

In [ ]:
plt.boxplot(X[:, 5])

##**4.3 - Escalado de Variables con dominios diferentes**

In [ ]:
# Verificamos si en nuestras variables numericas existen dominios distintos
# es decir rangos de valores minimo maximo muy diferentes en las divers variables

df_Xs.describe()

# Se detecan rangos muy diferentes, por lo que se procede a escalar

###**4.3.1 - Escalado basado en método estadistico de Estandarización**

In [ ]:
from sklearn.preprocessing import StandardScaler
#traemos esta libreria

In [ ]:
#crea un objeto escalador
obj_sc = StandardScaler()

In [ ]:
#aqui estamos escalado las variables
# obj_arr_np_X_sc

obj_X_sc = obj_sc.fit_transform( X )

In [ ]:
# Los valores promedios que empleo para escalar
obj_sc.scale_

In [ ]:
# Mostramos los valores de la varianza de cada variable que se empleo para escalar
obj_sc.var_  #

In [ ]:
type(obj_X_sc)

In [ ]:
obj_X_sc

###**4.3.2 - Escalado basado en método estadistico de Normalización**

In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
ob_sc_norm = MinMaxScaler()

X_sc_nor = ob_sc_norm.fit_transform(X)

In [ ]:
print(X_sc_nor)

# 6 - Dividir los datos en Entrenamiento (train) en Pruebas (test)

# 7 - Crear el Modelo

# 8 - Entrenar el Modelo

# 9 - Evaluar el modelo con metricas de calidad

# 10 - Hacer pruebas de laboratorio para verificar el funcionamiento del modelo